## Reconstrução de Base - People Analytics
Nesse projeto meu objetivo é **reconstruir a Base de dados People Analytics** que contém informações fictícias de colaboradores mês a mês. A reconstrução se faz necessária em função da **inconsistência** de alguns dados como por exemplo registros de colaboradores anterior a sua contratação ou posterior ao desligamento ou lacunas de registros entre data de admissão e desligamento.

Vou usar os seguintes passos para chegar no objetivo de reconstrução da base:
1.	Definir o problema;
2.	Coletar os dados e realizar um overview;
3.	Definir como vou replicar os registros para cada colaborador;
4.	Realizar a busca das informações necessárias com base na Data de Referência mais recente disponível para cada colaborador;
5.	Realizar a busca das informações que não mudam no tempo faltantes para cada colaborador;
6.	Concluir a recriação da Base.
#### 1. Definir o problema
Em uma análise inicial da Base fornecida foram encontradas algumas inconsistências como por exemplo:
- Colaborador ID 10219:
	- Data Referência = 11/2021 - 12/2024;
    - Data Admissão = 12/2019;
    - Data Desligamento = Ativo até 12/2024;
    - **De 12/2019 a 10/2021 não há registros.**
- Colaborador ID 10052:
	- Data Referência = 01/2021 - 12/2024;
    - Data Admissão = 03/2022;
    - Data Desligamento = Ativo até 12/2024;
    - **De 01/2021 a 02/2022 há registros extras.**
- Colaborador ID 10142:
	- Data Referência = 07/2023 - 12/2024;
    - Data Admissão = 01/2019;
    - Data Desligamento = 04/2019;
    - **De 07/2023 a 12/2024 há registros extras e de 01/2019 a 04/2019 há registros faltantes.**

Esses tipos de **inconsistências prejudicam as análises** necessárias da área de negócio. Estudos de Headcount, desligamentos e salário por período por exemplo resultariam em **valores que não correspondem a realidade**. Uma opção seria excluir tais registros inconsistentes por meio de filtros, contudo, tais **simplificações na base podem maquiar a realidade das análises**.

- **Qual o obtivo do projeto?**
    1. Reconstruir a Base de dados replicando as linhas uma quantidade de vezes igual a diferença de meses entre a Data de Desligamento e a Data de Admissão* (levando em conta que as datas de admissão e desligamento dos colaboradores estão corretas);
    2. Buscar as informações que variam no tempo com base na data de referência disponível mais próxima da data de referência recriada;
    3. Buscar as informações faltante com base no próprio ID do colaborador (informações que não variam no tempo).

- **Qual o benefício desse projeto?**
    1. Análises alinhadas com a realidade;
    2. Atendimento das necessidades da área de negócio
    3. Criação de um processo replicável para análises futuras

#### Importação das bibliotecas

In [1]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset

#### 2.	Coletar os dados e realizar um overview
- O Dataset foi disponibilizado pelos Curso de Pós Graduação em Análise de dados e IA Voltada para Negócios da Hashtag Treinamentos em Parceria coma Faculdade FNAT.
- Contém dados fictícios de colaboradores com informações de Cargo, área, salário e localidade onde a chave principal para diferenciar os colaboradores é a coluna “ID_Funcionario”.

In [2]:
df_original = pd.read_excel("../data/raw/Base_People_Analytics.xlsx")

In [3]:
display(df_original)

,Data_Referencia,ID_Funcionario,Nome_Funcionario,Departamento,Area,Cargo,Nivel,Estado,Tipo_Unidade,Salario,Faixa_Salarial,Data_Admissao,Data_Desligamento,Status_Emprego,Motivo_Desligamento,Fonte_Recrutamento,Pontuacao_Desempenho
0,2019-01-01,10132,Bruno Silva,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,7781,5k–8k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,LinkedIn,Atende plenamente
1,2019-02-01,10132,Bruno Silva,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,7781,5k–8k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,LinkedIn,Atende plenamente
2,2019-03-01,10132,Bruno Silva,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,7781,5k–8k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,LinkedIn,Atende plenamente
3,2019-04-01,10132,Bruno Silva,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,7781,5k–8k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,LinkedIn,Atende plenamente
4,2019-05-01,10132,Bruno Silva,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,7781,5k–8k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,LinkedIn,Atende plenamente
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19821,2022-04-01,10409,Camila Pereira,Produção,Operações,Técnico(a) de Produção I,Operacional,RJ,Filial,5891,5k–8k,2021-06-07,2022-11-16,Desligamento voluntário,Outra oportunidade,Candidatura online (site),Atende plenamente
19822,2022-05-01,10409,Camila Pereira,Produção,Operações,Técnico(a) de Produção I,Operacional,RJ,Filial,5891,5k–8k,2021-06-07,2022-11-16,Desligamento voluntário,Outra oportunidade,Candidatura online (site),Atende plenamente
19823,2022-06-01,10409,Camila Pereira,Produção,Operações,Técnico(a) de Produção I,Operacional,RJ,Filial,5891,5k–8k,2021-06-07,2022-11-16,Desligamento voluntário,Outra oportunidade,Candidatura online (site),Atende plenamente
19824,2022-07-01,10409,Camila Pereira,Produção,Operações,Técnico(a) de Produção I,Operacional,RJ,Filial,5891,5k–8k,2021-06-07,2022-11-16,Desligamento voluntário,Outra oportunidade,Candidatura online (site),Atende plenamente


| Coluna             | Tipo                | Descrição                                                                                  |
|--------------------|---------------------|--------------------------------------------------------------------------------------------|
| Data_Referencia    | Texto (dd/mm/aaaa)  | Primeiro dia do mês de referência (Jan/2019 a Dez/2024).                                  |
| ID_Funcionario     | Inteiro             | Código único do funcionário (chave natural).                                               |
| Nome_Funcionario   | Texto               | Nome completo do funcionário.                                                              |
| Departamento       | Texto               | Departamento (Produção, TI, Administração, Vendas, Diretoria, Engenharia de Software).     |
| Area               | Texto               | Área de agrupamento do departamento (Operações, Tecnologia, Administrativo, Comercial, Executivo). |
| Cargo              | Texto               | Cargo ocupado pelo funcionário (31 cargos distintos).                                      |
| Nivel              | Texto               | Nível hierárquico do cargo (Operacional, Especialista, Gerencial).                         |
| Estado             | Texto               | UF de lotação (SP, RJ, MG, BA).                                                            |
| Tipo_Unidade       | Texto               | Classificação da unidade (Matriz = SP, Filial = RJ/MG/BA).                                 |
| Salario            | Inteiro             | Salário mensal em reais (R$).                                                              |
| Faixa_Salarial     | Texto               | Faixa salarial calculada (0–5k, 5k–8k, 8k–12k, 12k–20k, 20k–35k, 35k–50k, 50k+).           |
| Data_Admissao      | Texto (dd/mm/aaaa)  | Data de admissão do funcionário na empresa.                                                |
| Data_Desligamento  | Texto (dd/mm/aaaa)  | Data de desligamento (vazio se funcionário ativo).                                         |
| Status_Emprego     | Texto               | Status atual: Ativo, Desligamento voluntário ou Desligado por justa causa.                |
| Motivo_Desligamento| Texto               | Motivo do desligamento (18 motivos distintos, ou 'N/A -Ainda empregado').                  |
| Fonte_Recrutamento | Texto               | Canal pelo qual o funcionário foi recrutado (LinkedIn, Indeed, Indicação, etc.).          |
| Pontuacao_Desempenho | Texto            | Avaliação de desempenho (Acima do esperado, Atende plenamente, Precisa melhorar, Plano de melhoria). |

Inspeção Geral da Base

In [4]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19826 entries, 0 to 19825
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Data_Referencia       19826 non-null  datetime64[ns]
 1   ID_Funcionario        19826 non-null  int64         
 2   Nome_Funcionario      19826 non-null  object        
 3   Departamento          19826 non-null  object        
 4   Area                  19826 non-null  object        
 5   Cargo                 19826 non-null  object        
 6   Nivel                 19826 non-null  object        
 7   Estado                19826 non-null  object        
 8   Tipo_Unidade          19826 non-null  object        
 9   Salario               19826 non-null  int64         
 10  Faixa_Salarial        19826 non-null  object        
 11  Data_Admissao         19826 non-null  datetime64[ns]
 12  Data_Desligamento     3674 non-null   datetime64[ns]
 13  Status_Emprego  

In [5]:
print(f'A base tem {df_original.shape[0]} linhas e {df_original.shape[1]} colunas')

A base tem 19826 linhas e 17 colunas


Alguns Insights:
1.	Com exceção da Coluna “Data_Desligamento”, todas as colunas possuem as 19.826 linhas preenchidas
2.	As linhas vazias da coluna “Data_Desligamento” são para colaboradores que ainda estão ativos

#### 3.	Definir como vou replicar os registros para cada colaborador

Para recriar a base, é necessário que, para cada colaborador, seu registro seja replicado o número de vezes da diferença entre o mês da data de desligamento e o mês da data de admissão, por exemplo:
- Data Contratação: 12/05/2021
- Data Desligamento: 05/09/2021

Neste caso Deve haver 5 registros para este colaborador (referentes aos meses de 05/2021, 06/2021, 07/2021, 08/2021 e 09/2021)

**Obs:** para colaboradores que ainda são ativos, deve-se adotar a Data de Desligamento como sendo a última data de referência da base (nesse caso 12/2024).

Para realizar essa replicação, necessito apenas do ID do colaborador, Data de Admissão e Data de Desligamento.
Será criada um novo df apenas com tais colunas (df_aux).

Além, disso, como explanado acima, os colaboradores ainda ativos terão sua Data de Desligamento preenchido com a data de 01/12/2024.

In [6]:
df_aux = df_original[["ID_Funcionario", "Data_Admissao", "Data_Desligamento"]].copy()
df_aux["Data_Desligamento"] = df_aux["Data_Desligamento"].fillna(pd.Timestamp("2024-12-01"))

Como o objetivo é replicar as linhas, vou manter apenas registros únicos para cada colaborador usando o ID do Colaborador como chave única

In [7]:
df_aux = df_aux.drop_duplicates(subset=["ID_Funcionario"]).sort_values("ID_Funcionario")

In [8]:
display(df_aux)

,ID_Funcionario,Data_Admissao,Data_Desligamento
14856,10001,2019-01-01,2024-12-01
2108,10002,2021-12-29,2024-12-01
2321,10003,2020-09-24,2024-12-01
19531,10004,2019-11-26,2023-01-13
16482,10005,2022-10-17,2024-09-08
...,...,...,...
9018,10496,2021-09-17,2024-12-01
4842,10497,2019-01-01,2024-12-01
7161,10498,2019-03-13,2024-12-01
1042,10499,2019-02-08,2024-12-01


Criando a coluna que conterá a quantidade de vezes que cada linhas deverá ser replicada com base na quantidade de meses entre Admissão e Desligamento

In [9]:
meses_diff = (
    df_aux["Data_Desligamento"].dt.to_period("M").astype("int64")
    - df_aux["Data_Admissao"].dt.to_period("M").astype("int64") + 1
)

df_aux["Meses_Diferenca"] = meses_diff

In [10]:
display(df_aux)

,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca
14856,10001,2019-01-01,2024-12-01,72
2108,10002,2021-12-29,2024-12-01,37
2321,10003,2020-09-24,2024-12-01,52
19531,10004,2019-11-26,2023-01-13,39
16482,10005,2022-10-17,2024-09-08,24
...,...,...,...,...
9018,10496,2021-09-17,2024-12-01,40
4842,10497,2019-01-01,2024-12-01,72
7161,10498,2019-03-13,2024-12-01,70
1042,10499,2019-02-08,2024-12-01,71


In [11]:
print(f'O df resultante deverá ter {sum(df_aux["Meses_Diferenca"])} linhas')

O df resultante deverá ter 20284 linhas


Replicando o df N vezes, onde N é a diferença de meses

In [12]:
# repetir cada linha N vezes (N = Meses_Diferenca) já resetando o index
df_aux = df_aux.loc[
    df_aux.index.repeat(df_aux["Meses_Diferenca"])
].reset_index(drop=True)

In [13]:
display(df_aux)
print(f'A Base resultante contém {df_aux.shape[0]} linhas, em acordo com o que deveria ser, conforme calculado anterirormente')

,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca
0,10001,2019-01-01,2024-12-01,72
1,10001,2019-01-01,2024-12-01,72
2,10001,2019-01-01,2024-12-01,72
3,10001,2019-01-01,2024-12-01,72
4,10001,2019-01-01,2024-12-01,72
...,...,...,...,...
20279,10500,2022-03-21,2024-12-01,34
20280,10500,2022-03-21,2024-12-01,34
20281,10500,2022-03-21,2024-12-01,34
20282,10500,2022-03-21,2024-12-01,34


A Base resultante contém 20284 linhas, em acordo com o que deveria ser, conforme calculado anterirormente


Verificando a quantidade de Recorrências criadas para casos anteriormente falhos

In [14]:
print(f'O Colaborador de ID 10142 deveria ter {len(df_aux[df_aux["ID_Funcionario"] == 10142])} registros')
display(df_aux[df_aux["ID_Funcionario"] == 10142])

print(f'O Colaborador de ID 10052 deveria ter {len(df_aux[df_aux["ID_Funcionario"] == 10052])} registros')
display(df_aux[df_aux["ID_Funcionario"] == 10052])

print(f'O Colaborador de ID 10219 deveria ter {len(df_aux[df_aux["ID_Funcionario"] == 10219])} registros')
display(df_aux[df_aux["ID_Funcionario"] == 10219])

O Colaborador de ID 10142 deveria ter 4 registros


,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca
5899,10142,2019-01-01,2019-04-05,4
5900,10142,2019-01-01,2019-04-05,4
5901,10142,2019-01-01,2019-04-05,4
5902,10142,2019-01-01,2019-04-05,4


O Colaborador de ID 10052 deveria ter 34 registros


,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca
2051,10052,2022-03-23,2024-12-01,34
2052,10052,2022-03-23,2024-12-01,34
2053,10052,2022-03-23,2024-12-01,34
2054,10052,2022-03-23,2024-12-01,34
2055,10052,2022-03-23,2024-12-01,34
2056,10052,2022-03-23,2024-12-01,34
2057,10052,2022-03-23,2024-12-01,34
2058,10052,2022-03-23,2024-12-01,34
2059,10052,2022-03-23,2024-12-01,34
2060,10052,2022-03-23,2024-12-01,34


O Colaborador de ID 10219 deveria ter 61 registros


,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca
8772,10219,2019-12-20,2024-12-01,61
8773,10219,2019-12-20,2024-12-01,61
8774,10219,2019-12-20,2024-12-01,61
8775,10219,2019-12-20,2024-12-01,61
8776,10219,2019-12-20,2024-12-01,61
...,...,...,...,...
8828,10219,2019-12-20,2024-12-01,61
8829,10219,2019-12-20,2024-12-01,61
8830,10219,2019-12-20,2024-12-01,61
8831,10219,2019-12-20,2024-12-01,61


Criando um Runningsum que vai servir como base para recriação da coluna de “Data_Referencia”

In [15]:
df_aux["Num_Repeticao"] = (
    df_aux.groupby("ID_Funcionario")
      .cumcount() + 1          # começa em 1
)

In [16]:
display(df_aux)

,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca,Num_Repeticao
0,10001,2019-01-01,2024-12-01,72,1
1,10001,2019-01-01,2024-12-01,72,2
2,10001,2019-01-01,2024-12-01,72,3
3,10001,2019-01-01,2024-12-01,72,4
4,10001,2019-01-01,2024-12-01,72,5
...,...,...,...,...,...
20279,10500,2022-03-21,2024-12-01,34,30
20280,10500,2022-03-21,2024-12-01,34,31
20281,10500,2022-03-21,2024-12-01,34,32
20282,10500,2022-03-21,2024-12-01,34,33


Criando e aplicando a função que Preenche a nova coluna de “Data_Referencia”.

Essa função além de somar ao mês de data de admissão o número de messes necessário para a “Data_Referencia”, também cria tal data já no dia um de cada mês, seguindo o padrão inicialmente imposto pela base inicial.

In [17]:
# Definindo a função que vai preencher a nova data de refrencia
def calcula_data(row):
    # admissão + (N-1) meses
    data = row["Data_Admissao"] + DateOffset(months=int(row["Num_Repeticao"] - 1))
    # força para o 1º dia do mês
    return data.to_period("M").to_timestamp()

# Aplicando a função liinhas a linhas
df_aux["Data_Referencia"] = df_aux.apply(calcula_data, axis=1)

In [18]:
display(df_aux)

,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca,Num_Repeticao,Data_Referencia
0,10001,2019-01-01,2024-12-01,72,1,2019-01-01
1,10001,2019-01-01,2024-12-01,72,2,2019-02-01
2,10001,2019-01-01,2024-12-01,72,3,2019-03-01
3,10001,2019-01-01,2024-12-01,72,4,2019-04-01
4,10001,2019-01-01,2024-12-01,72,5,2019-05-01
...,...,...,...,...,...,...
20279,10500,2022-03-21,2024-12-01,34,30,2024-08-01
20280,10500,2022-03-21,2024-12-01,34,31,2024-09-01
20281,10500,2022-03-21,2024-12-01,34,32,2024-10-01
20282,10500,2022-03-21,2024-12-01,34,33,2024-11-01


#### 4.	Realizar a busca das informações necessárias com base na Data de Referência mais recente disponível para cada colaborador
Agora, com as Datas de Referência recriadas e preenchidas de forma correta para cada colaborador refletindo exatamente o período em que esteve ativo, vamos buscar na base original as informações de “Salario” e “Status_Emprego” mais próximas com base na Data de Referência.

**Obs:** Essas são as duas únicas informações buscadas dessa maneira porque são as únicas que variam no tempo para um mesmo colaborador.


Para realizar essa busca será usada a função merge_asof que obriga ambas as colunas de dadas usadas estejam organizadas por ordem de data antes de fazer uso da função propriamente dita.

In [19]:
df_original = df_original.sort_values("Data_Referencia")
df_aux = df_aux.sort_values("Data_Referencia")

df_aux = pd.merge_asof(
    df_aux,
    df_original[["ID_Funcionario", "Data_Referencia", "Salario", "Status_Emprego"]],
    on="Data_Referencia",              # coluna temporal
    by="ID_Funcionario",               # agrupa por colaborador (2ª condição)
    direction="nearest"                # pega a data mais próxima (antes ou depois)
)

Reorganizando o df com base no ID do Colaborador e na Data de Referência além de resetar a coluna de Indexador

In [20]:
df_aux = df_aux.sort_values(["ID_Funcionario", "Data_Admissao"]).reset_index(drop=True)

Verificando alguns resultados

In [21]:
display(df_aux.loc[df_aux['ID_Funcionario'] == 10142].tail(4))
display(df_aux.loc[df_aux['ID_Funcionario'] == 10130].iloc[6:10])
display(df_aux.loc[df_aux['ID_Funcionario'] == 10219].tail(3))

,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca,Num_Repeticao,Data_Referencia,Salario,Status_Emprego
5899,10142,2019-01-01,2019-04-05,4,1,2019-01-01,18612,Desligado por justa causa
5900,10142,2019-01-01,2019-04-05,4,2,2019-02-01,18612,Desligado por justa causa
5901,10142,2019-01-01,2019-04-05,4,3,2019-03-01,18612,Desligado por justa causa
5902,10142,2019-01-01,2019-04-05,4,4,2019-04-01,18612,Desligado por justa causa


,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca,Num_Repeticao,Data_Referencia,Salario,Status_Emprego
5278,10130,2023-05-06,2024-12-31,20,7,2023-11-01,20497,Desligamento voluntário
5279,10130,2023-05-06,2024-12-31,20,8,2023-12-01,20497,Desligamento voluntário
5280,10130,2023-05-06,2024-12-31,20,9,2024-01-01,21720,Desligamento voluntário
5281,10130,2023-05-06,2024-12-31,20,10,2024-02-01,21720,Desligamento voluntário


,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca,Num_Repeticao,Data_Referencia,Salario,Status_Emprego
8830,10219,2019-12-20,2024-12-01,61,59,2024-10-01,6184,Ativo
8831,10219,2019-12-20,2024-12-01,61,60,2024-11-01,6184,Ativo
8832,10219,2019-12-20,2024-12-01,61,61,2024-12-01,6184,Ativo


Os resultados acima mostram o sucesso na busca por proximidade de data, tanto para o Salário quanto o Status de colaborador

Agora, com as informações corretas, devemos corrigir o “Status_Emprego” para que nos meses em que o colaborador estava ativo ele conste como ativo e não como a razão de seu eventual desligamento

In [22]:
# onde o Mês da Data_Referencia for diferente Data_Desligamento → "Ativo"
mask_mes_diferente = (
    df_aux["Data_Referencia"].dt.to_period("M")
    != df_aux["Data_Desligamento"].dt.to_period("M")
)

df_aux.loc[mask_mes_diferente, "Status_Emprego"] = "Ativo"

Verificando a correção dos registros nas datas em que os colaboradores estavam ativos e a quantidade total de colaboradores desligados

In [23]:
display(df_aux.loc[df_aux['ID_Funcionario'] == 10142])

display(f'A base deve conter {df_aux.loc[df_aux["Status_Emprego"] != "Ativo"].shape[0]} colaboradores que foram desligados')
display(df_aux.loc[df_aux["Status_Emprego"] != "Ativo"])

,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca,Num_Repeticao,Data_Referencia,Salario,Status_Emprego
5899,10142,2019-01-01,2019-04-05,4,1,2019-01-01,18612,Ativo
5900,10142,2019-01-01,2019-04-05,4,2,2019-02-01,18612,Ativo
5901,10142,2019-01-01,2019-04-05,4,3,2019-03-01,18612,Ativo
5902,10142,2019-01-01,2019-04-05,4,4,2019-04-01,18612,Desligado por justa causa


'A base deve conter 155 colaboradores que foram desligados'

,ID_Funcionario,Data_Admissao,Data_Desligamento,Meses_Diferenca,Num_Repeticao,Data_Referencia,Salario,Status_Emprego
199,10004,2019-11-26,2023-01-13,39,39,2023-01-01,6404,Desligamento voluntário
223,10005,2022-10-17,2024-09-08,24,24,2024-09-01,5367,Desligamento voluntário
654,10014,2023-08-27,2024-08-01,13,13,2024-08-01,5994,Desligamento voluntário
911,10022,2023-05-17,2024-12-31,20,20,2024-12-01,7806,Desligado por justa causa
1235,10030,2021-07-11,2022-04-30,10,10,2022-04-01,4145,Desligamento voluntário
...,...,...,...,...,...,...,...,...
19244,10475,2019-01-01,2019-01-01,1,1,2019-01-01,7224,Desligamento voluntário
19271,10476,2021-04-15,2023-06-24,27,27,2023-06-01,4446,Desligamento voluntário
19278,10477,2019-01-02,2019-07-28,7,7,2019-07-01,7073,Desligamento voluntário
19848,10491,2023-05-12,2024-06-16,14,14,2024-06-01,4146,Desligamento voluntário


#### 5.	Realizar a busca das informações que não mudam no tempo faltantes para cada colaborador

Primeiramente vamos criar um no df que vai conter apenas as colunas úteis provenientes da df_aux que estarão do df final

In [24]:
df_Base_People_Analytics = df_aux[['Data_Referencia', 'ID_Funcionario', 'Salario', 'Status_Emprego']].copy()

Agora para trazer as colunas restante para composição no df final, precisamos criar uma tabela auxiliar que conterá apenas os valores únicos de “ID_Funcionario” e deverá estar organizada por "ID_Funcionario" e "Data_Referencia"

In [25]:
# Trazendo os valores das outras colunas da base original
# Selecionando as colunas
cols_dim = [
    "ID_Funcionario", "Nome_Funcionario", "Departamento", "Area", "Cargo", "Nivel", "Estado", "Tipo_Unidade", "Faixa_Salarial", "Data_Admissao", "Data_Desligamento", "Motivo_Desligamento", "Fonte_Recrutamento", "Pontuacao_Desempenho"
    ]

# ordenar para definir qual linha será a referência (ex.: menor Data_Referencia)
df_dim = (
    df_original
    .sort_values(["ID_Funcionario", "Data_Referencia"])
    .loc[:, cols_dim]
    .drop_duplicates(subset="ID_Funcionario", keep="first")
)

Trazendo os valores das colunas faltantes da Base Original

In [26]:
# trazendo os valores da tabela original (equivalente procv)
df_Base_People_Analytics = df_Base_People_Analytics.merge(
    df_dim,
    on="ID_Funcionario",
    how="left"
)

#### 6.	Concluir a recriação da Base

Reorganizado as colunas da base final para coincidir com a Base Orginal

In [27]:
df_Base_People_Analytics = df_Base_People_Analytics[['Data_Referencia', 'ID_Funcionario', 'Nome_Funcionario', 'Departamento', 'Area', 'Cargo', 'Nivel', 'Estado', 'Tipo_Unidade', 'Salario', 'Faixa_Salarial', 'Data_Admissao', 'Data_Desligamento','Status_Emprego', 'Motivo_Desligamento', 'Fonte_Recrutamento', 'Pontuacao_Desempenho']]

display(df_Base_People_Analytics.shape)
display(df_Base_People_Analytics)

(20284, 17)

,Data_Referencia,ID_Funcionario,Nome_Funcionario,Departamento,Area,Cargo,Nivel,Estado,Tipo_Unidade,Salario,Faixa_Salarial,Data_Admissao,Data_Desligamento,Status_Emprego,Motivo_Desligamento,Fonte_Recrutamento,Pontuacao_Desempenho
0,2019-01-01,10001,Bruno Ferreira,Produção,Operações,Gerente de Produção,Gerencial,MG,Filial,28393,20k–35k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,Indeed,Acima do esperado
1,2019-02-01,10001,Bruno Ferreira,Produção,Operações,Gerente de Produção,Gerencial,MG,Filial,28393,20k–35k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,Indeed,Acima do esperado
2,2019-03-01,10001,Bruno Ferreira,Produção,Operações,Gerente de Produção,Gerencial,MG,Filial,28393,20k–35k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,Indeed,Acima do esperado
3,2019-04-01,10001,Bruno Ferreira,Produção,Operações,Gerente de Produção,Gerencial,MG,Filial,28393,20k–35k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,Indeed,Acima do esperado
4,2019-05-01,10001,Bruno Ferreira,Produção,Operações,Gerente de Produção,Gerencial,MG,Filial,28393,20k–35k,2019-01-01,NaT,Ativo,N/A - Ainda empregado,Indeed,Acima do esperado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20279,2024-08-01,10500,João Costa,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,6400,5k–8k,2022-03-21,NaT,Ativo,N/A - Ainda empregado,Indeed,Atende plenamente
20280,2024-09-01,10500,João Costa,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,6400,5k–8k,2022-03-21,NaT,Ativo,N/A - Ainda empregado,Indeed,Atende plenamente
20281,2024-10-01,10500,João Costa,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,6400,5k–8k,2022-03-21,NaT,Ativo,N/A - Ainda empregado,Indeed,Atende plenamente
20282,2024-11-01,10500,João Costa,Produção,Operações,Técnico(a) de Produção I,Operacional,SP,Matriz,6400,5k–8k,2022-03-21,NaT,Ativo,N/A - Ainda empregado,Indeed,Atende plenamente


Concluindo a recriação em formato Excel

In [29]:
df_Base_People_Analytics.to_excel("Base_People_Analytics_refeita.xlsx", sheet_name="BaseRH", index=False)